# Retrieval evaluation with manual relevance judgments (MiniLM vs MPNet)

This notebook evaluates retrieval quality of our backend **/search** endpoint.

We measure how well the system retrieves relevant academic papers for a set of test queries.

The evaluation is designed to be:
- **Reproducible** (results are saved to CSV)
- **Fair** (the same labeled dataset is used for both models)

Relevance Labels:
Each (query, document) pair is manually labeled:

- **label = 1** -> the paper is relevant to the query  
- **label = 0** -> the paper is not relevant to the query  

A paper is considered relevant if it directly addresses the query topic.

Labels are assigned manually based on:
- paper title
- abstract
- and topic relevance

Key for matching labels:
(query, abs_url)

We compare two embedding models:
- **MiniLM** - smaller and faster
- **MPNet** — larger and potentially more accurate

In [ ]:
# Configuration

from datetime import datetime

BACKEND_URL = "http://127.0.0.1:8000"
SEARCH_ENDPOINT = f"{BACKEND_URL}/search"

TIMEOUT_S = 30

# Set this manually for each experiment run.
# Use the same model name we used when we reindexed the backend.
# MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

# Retrieval settings
TOP_K = 10
ALPHAS = [0.0, 0.7, 1.0]   # lexical, hybrid, semantic

# Keep this query list stable across runs (MiniLM vs MPNet comparisons)
QUERIES = [
    "post hoc explanation",
    "model interpretability",
    "counterfactual explanation",
    "feature attribution",
    "explainable ai evaluation",
    "human-centered explainable ai",
    "causal explanation",
    "fairness explanation",
    "model debugging",
    "explanation methods",
]

# Output files (timestamped, so we never overwrite by accident)
RUN_ID = datetime.now().strftime("%Y-%m-%d_%H%M%S")
RESULTS_CSV = f"retrieval_eval_results_{MODEL_NAME.replace('/', '_')}_{RUN_ID}.csv"
POOL_CSV = f"pool_to_label_{MODEL_NAME.replace('/', '_')}_{RUN_ID}.csv"

# We create this file by copying POOL_CSV and filling label (0/1)
MANUAL_JUDGMENTS_CSV = "manual_relevance_judgments.csv"

In [ ]:
import requests
import pandas as pd
import numpy as np
from typing import Any, Dict, List

pd.set_option("display.max_colwidth", 120)

This section defines helper functions that are used throughout the evaluation process.

These functions are responsible for:
- Sending search requests to the backend API (POST /search)
- Safely converting API fields to strings
- Computing evaluation metrics (Precision@K)

They only support querying, parsing, and metric computation.

In [ ]:
# Helpers
def _post_search(query: str, top_k: int, alpha: float, timeout_s: int = 30) -> Dict[str, Any]:
    """Call POST /search (read-only)."""
    payload = {"query": query, "top_k": int(top_k), "alpha": float(alpha)}
    r = requests.post(SEARCH_ENDPOINT, json=payload, timeout=timeout_s)
    r.raise_for_status()
    return r.json()

def _to_str(x: Any) -> str:
    """Convert a field to a safe string (handles list/None)."""
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, (list, tuple)):
        return ", ".join([str(i) for i in x if i is not None])
    return str(x)

def precision_at_k(ranked_labels: List[int], k: int) -> float:
    """Precision@K = relevant_in_topK / K."""
    if k <= 0:
        return np.nan
    top = ranked_labels[:k]
    if len(top) == 0:
        return np.nan
    return float(np.sum(top)) / float(len(top))

## Run searches and save a results log

This section queries the backend search API for each (query, alpha) pair and stores a detailed results log in CSV format.

The log is useful even before labeling (debug + transparency).

In [34]:
rows: List[Dict[str, Any]] = []

for q in QUERIES:
    for a in ALPHAS:
        data = _post_search(q, top_k=TOP_K, alpha=a, timeout_s=TIMEOUT_S)
        items = data.get("items", []) or []

        for rank, it in enumerate(items, start=1):
            explain = it.get("explain", {}) or {}

            # Chunk info (source-level)
            chunk_index = explain.get("chunk_index", None)
            start_token = explain.get("start_token", None)
            end_token = explain.get("end_token", None)

            rows.append({
                "run_id": RUN_ID,
                "model": MODEL_NAME,
                "query": q,
                "alpha": float(a),
                "top_k": int(TOP_K),
                "rank": int(rank),

                # Stable document key for labeling / dedupe
                "abs_url": _to_str(it.get("abs_url")),
                "pdf_url": _to_str(it.get("pdf_url")),

                # Display fields
                "title": _to_str(it.get("title")),
                "authors": _to_str(it.get("authors")),
                "published": _to_str(it.get("published")),

                # Chunk info (logged for analysis)
                "chunk_index": chunk_index,
                "start_token": start_token,
                "end_token": end_token,

                # Final score (hybrid)
                "final_score": it.get("score", None),

                # Explain fields: components used by alpha-weighted scoring
                "semantic_component": explain.get("semantic_component", None),
                "lexical_component": explain.get("lexical_component", None),
            })

df = pd.DataFrame(rows)

# Basic numeric cleanup
for col in ["final_score", "semantic_component", "lexical_component", "chunk_index", "start_token", "end_token"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["rank"] = pd.to_numeric(df["rank"], errors="coerce").astype("Int64")

df.to_csv(RESULTS_CSV, index=False)
print(f"Saved results log: {RESULTS_CSV}  (rows={len(df)})")

df.head(10)

Saved results log: retrieval_eval_results_sentence-transformers_all-mpnet-base-v2_2026-01-28_150412.csv  (rows=300)


,run_id,model,query,alpha,top_k,rank,abs_url,pdf_url,title,authors,published,chunk_index,start_token,end_token,final_score,semantic_component,lexical_component
0,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,1,https://arxiv.org/abs/2206.14841v1,https://arxiv.org/pdf/2206.14841v1,Causality for Inherently Explainable Transformers: CAT-XPLAIN,"Subash Khanal, Benjamin Brodie, Xin Xing, Ai-Ling Lin, Nathan Jacobs",2022-06-29T18:11:01Z,0,NaN,NaN,1.00,0.82,1.00
1,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,2,https://arxiv.org/abs/2407.19897v1,https://arxiv.org/pdf/2407.19897v1,BEExAI: Benchmark to Evaluate Explainable AI,"Samuel Sithakoul, Sara Meftah, Clément Feutry",2024-07-29T11:21:17Z,0,NaN,NaN,0.82,0.81,0.82
2,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,3,https://arxiv.org/abs/2307.09673v3,https://arxiv.org/pdf/2307.09673v3,What's meant by explainable model: A Scoping Review,"Mallika Mainali, Rosina O Weber",2023-07-18T22:55:04Z,0,NaN,NaN,0.71,0.97,0.71
3,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,4,https://arxiv.org/abs/2308.03589v1,https://arxiv.org/pdf/2308.03589v1,Feature Importance versus Feature Influence and What It Signifies for Explainable AI,Kary Främling,2023-08-07T13:46:18Z,0,NaN,NaN,0.65,1.00,0.65
4,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,5,https://arxiv.org/abs/2408.04746v1,https://arxiv.org/pdf/2408.04746v1,More Questions than Answers? Lessons from Integrating Explainable AI into a Cyber-AI Tool,"Ashley Suh, Harry Li, Caitlin Kenney, Kenneth Alperin, Steven R. Gomez",2024-08-08T20:09:31Z,0,NaN,NaN,0.57,0.00,0.57
5,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,6,https://arxiv.org/abs/2401.13334v3,https://arxiv.org/pdf/2401.13334v3,Explainable Bayesian Optimization,"Tanmay Chakraborty, Christian Wirth, Christin Seifert",2024-01-24T09:59:22Z,0,NaN,NaN,0.56,0.00,0.56
6,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,7,https://arxiv.org/abs/2506.02262v1,https://arxiv.org/pdf/2506.02262v1,Composable Building Blocks for Controllable and Transparent Interactive AI Systems,"Sebe Vanbrabant, Gustavo Rovelo Ruiz, Davy Vanacken",2025-06-02T21:10:51Z,0,NaN,NaN,0.44,0.00,0.44
7,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,8,https://arxiv.org/abs/2509.14830v2,https://arxiv.org/pdf/2509.14830v2,ProtoMedX: Towards Explainable Multi-Modal Prototype Learning for Bone Health Classification,"Alvaro Lopez Pellicer, Andre Mariucci, Plamen Angelov, Marwan Bukhari, Jemma G. Kerns",2025-09-18T10:46:18Z,0,NaN,NaN,0.42,0.00,0.42
8,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,9,https://arxiv.org/abs/2007.04477v13,https://arxiv.org/pdf/2007.04477v13,Good AI for the Present of Humanity Democratizing AI Governance,"Nicholas Kluge Corrêa, Nythamar de Oliveira",2020-07-08T23:50:28Z,0,NaN,NaN,0.21,0.00,0.21
9,2026-01-28_150412,sentence-transformers/all-mpnet-base-v2,post hoc explanation,0.00,10,10,https://arxiv.org/abs/2406.00216v2,https://arxiv.org/pdf/2406.00216v2,The Explanation Necessity for Healthcare AI,"Michail Mamalakis, Héloïse de Vareilles, Graham Murray, Pietro Lio, John Suckling",2024-05-31T22:20:10Z,0,NaN,NaN,0.21,0.88,0.21


## Build a labeling pool (pooling)

We create a pool of unique (query, abs_url) pairs across all alpha settings.
This reduces bias (you do not label only one method's results).

Next step:
- Open POOL_CSV
- Fill label with 0/1
- Save as manual_relevance_judgments.csv

In [ ]:
pool = (
    df[["query", "abs_url", "title", "pdf_url"]]
    .dropna(subset=["abs_url"])
    .drop_duplicates(subset=["query", "abs_url"])
    .copy()
)

pool["label"] = ""  # we fill this manually with 0/1

pool = pool.sort_values(["query", "abs_url"]).reset_index(drop=True)

pool.to_csv(POOL_CSV, index=False)
print(f"Saved pool for labeling: {POOL_CSV}")
print("Create manual labels:")
print(f"1) Open {POOL_CSV}")
print("2) Fill label with 0 or 1")
print(f"3) Save as {MANUAL_JUDGMENTS_CSV} in the same folder")

pool.head(10)

Saved pool for labeling: pool_to_label_sentence-transformers_all-MiniLM-L6-v2_2026-01-27_210329.csv
Create manual labels:
1) Open pool_to_label_sentence-transformers_all-MiniLM-L6-v2_2026-01-27_210329.csv
2) Fill label with 0 or 1
3) Save as manual_relevance_judgments.csv in the same folder


,query,abs_url,title,pdf_url,label
0,causal explanation,https://arxiv.org/abs/1908.02619v1,Experiential AI,https://arxiv.org/pdf/1908.02619v1,
1,causal explanation,https://arxiv.org/abs/1909.06907v1,X-ToM: Explaining with Theory-of-Mind for Gaining Justified Human Trust,https://arxiv.org/pdf/1909.06907v1,
2,causal explanation,https://arxiv.org/abs/2003.07520v1,Foundations of Explainable Knowledge-Enabled Systems,https://arxiv.org/pdf/2003.07520v1,
3,causal explanation,https://arxiv.org/abs/2004.14545v2,Explainable Deep Learning: A Field Guide for the Uninitiated,https://arxiv.org/pdf/2004.14545v2,
4,causal explanation,https://arxiv.org/abs/2005.13275v1,Who is this Explanation for? Human Intelligence and Knowledge Graphs for eXplainable AI,https://arxiv.org/pdf/2005.13275v1,
5,causal explanation,https://arxiv.org/abs/2106.05568v2,"Explainable AI, but explainable to whom?",https://arxiv.org/pdf/2106.05568v2,
6,causal explanation,https://arxiv.org/abs/2108.05317v2,Model-agnostic vs. Model-intrinsic Interpretability for Explainable Product Search,https://arxiv.org/pdf/2108.05317v2,
7,causal explanation,https://arxiv.org/abs/2203.11547v1,Explainability in reinforcement learning: perspective and position,https://arxiv.org/pdf/2203.11547v1,
8,causal explanation,https://arxiv.org/abs/2205.01467v1,On the Effect of Information Asymmetry in Human-AI Teams,https://arxiv.org/pdf/2205.01467v1,
9,causal explanation,https://arxiv.org/abs/2206.14841v1,Causality for Inherently Explainable Transformers: CAT-XPLAIN,https://arxiv.org/pdf/2206.14841v1,


## Evaluation: comparing retrieval configurations

In this section we compare two index configurations built from the same document corpus. One chunked at 224 tokens, one at 360 tokens. Each index was searched in three modes: lexical only (BM25), hybrid, and semantic only.

**What we're comparing:**
- Two chunk sizes: 224-token and 360-token
- Three retrieval modes per index: BM25 (alpha=0.0), hybrid (alpha=0.7), semantic (alpha=1.0)
- All evaluated on the same 10 queries using a shared manual relevance judgment pool

**What the columns mean:**
- The 224-token and 360-token columns refer to the chunk size of the index, not to the embedding model
- At alpha=0.0 (BM25), no embeddings are used, but chunk size still affects results, because BM25 scores chunk-level text
- At alpha=1.0 (semantic), only embeddings are used. The chunk size determines how much context each chunk carries

**Metrics:**
- **Precision@K (dedup-adjusted):** precision is measured on the unique papers that remain after deduplicating chunks. When fewer than K papers are retrieved, the calculation is based on the available papers rather than a fixed K.
- **coverage@10:** fraction of the 10 positions that contain a judged document. All retrieved docs are judged in this dataset, so coverage = n_docs@10 / 10.
- **avg_docs@10:** average number of unique papers remaining in the top-10 after deduplicating chunks by abs_url. Values below 10 indicate that multiple top-ranked chunks came from the same paper.

**Limitation:** The current pipeline requests top_k=10 chunks, then deduplicates to unique papers. This results in 6-10 unique papers (avg 8.7 for 224-token, 9.6 for 360-token at hybrid).

**Future work:** request top_k=50 chunks to guarantee 10 unique papers.

In [62]:
pd.options.display.float_format = "{:.2f}".format

JUDGMENTS = "evaluation/manual_relevance_judgments_union_minilm_mpnet_with_abstracts.csv"

MINILM_RESULTS = "evaluation/retrieval_eval_results_sentence-transformers_all-MiniLM-L6-v2_2026-01-27_210329.csv"
MPNET_RESULTS  = "evaluation/retrieval_eval_results_sentence-transformers_all-mpnet-base-v2_2026-01-28_150412.csv"

TOP_K = 10

# Load judgments
judg = pd.read_csv(JUDGMENTS)[["query", "abs_url", "label"]].copy()
judg["query"] = judg["query"].astype(str).str.strip()
judg["abs_url"] = judg["abs_url"].astype(str).str.strip()

# keep only 0/1 as valid, everything else -> NaN (unjudged)
judg["label"] = pd.to_numeric(judg["label"], errors="coerce")
judg.loc[~judg["label"].isin([0, 1]), "label"] = np.nan

# Helpers
def load_results(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)[["query", "abs_url", "alpha", "rank"]].copy()
    df["query"] = df["query"].astype(str).str.strip()
    df["abs_url"] = df["abs_url"].astype(str).str.strip()
    df["alpha"] = pd.to_numeric(df["alpha"], errors="coerce")
    df["rank"]  = pd.to_numeric(df["rank"],  errors="coerce")
    df = df.dropna(subset=["query", "abs_url", "alpha", "rank"]).copy()
    df["rank"] = df["rank"].astype(int)
    df = df[df["rank"] <= TOP_K].copy()
    return df

def dedupe_doc_topk(g: pd.DataFrame) -> pd.DataFrame:
    # doc-level: keep best ranked chunk per abs_url, then take top_k docs
    return (
        g.sort_values("rank")
         .drop_duplicates("abs_url", keep="first")
         .head(TOP_K)
         .copy()
    )

# Precision@K over actually retrieved unique papers (after chunk->paper deduplication).
# If deduplication leaves fewer than K papers, divide by the available count (len(topk)), not by K.
# This is not a new metric: it applies the standard precision definition to the retrieved list.
def precision_at_k_available(top: pd.DataFrame, k: int) -> float:
    topk = top.head(k)
    if len(topk) == 0:
        return np.nan
    if topk["label"].isna().any():
        return np.nan
    return float(topk["label"].sum()) / float(len(topk))

def evaluate(results_csv: str, chunk_size_label: str):
    df = load_results(results_csv)
    merged = df.merge(judg, on=["query", "abs_url"], how="left")

    rows = []
    for (alpha, query), g in merged.groupby(["alpha", "query"]):
        top = dedupe_doc_topk(g)

        # coverage@10: fraction of the 10 positions that are judged
        coverage10 = float(top.head(10)["label"].notna().sum()) / 10.0

        rows.append({
            "chunk_size": chunk_size_label,
            "alpha": float(alpha),
            "query": query,
            "P@5":  precision_at_k_available(top, 5),
            "P@10": precision_at_k_available(top, 10),
            "coverage@10": coverage10,
            "n_docs@10": int(len(top)),  # unique docs after dedupe
        })

    per_query = pd.DataFrame(rows)
    summary = (
        per_query
        .groupby(["chunk_size", "alpha"], as_index=False)
        .agg(
            P_at_5=("P@5", "mean"),
            P_at_10=("P@10", "mean"),
            coverage_at_10=("coverage@10", "mean"),
            avg_docs=("n_docs@10", "mean"),
            min_docs=("n_docs@10", "min"),
        )
        .sort_values(["alpha", "chunk_size"])
        .reset_index(drop=True)
    )
    summary.columns = ["chunk_size","alpha","P@5","P@10","coverage@10","avg_docs@10","min_docs@10"]
    return per_query, summary


# Run evaluation
# chunk_size label: MiniLM run used 224-token chunks, MPNet run used 360-token chunks
minilm_perq, minilm_sum = evaluate(MINILM_RESULTS, "224-token")
mpnet_perq,  mpnet_sum  = evaluate(MPNET_RESULTS,  "360-token")

# Debug: show per-query results to check for NaN issues
print("\nQueries with < 10 unique docs (224-token):")
display(minilm_perq[minilm_perq["n_docs@10"] < 10])

print("\nQueries with < 10 unique docs (360-token):")
display(mpnet_perq[mpnet_perq["n_docs@10"] < 10])

print("\nAll queries sorted by n_docs (224-token):")
display(minilm_perq.sort_values("n_docs@10"))

print("\nAll queries sorted by n_docs (360-token):")
display(mpnet_perq.sort_values("n_docs@10"))

print("224-token index summary (MiniLM run):")
display(minilm_sum)

print("\n360-token index summary (MPNet run):")
display(mpnet_sum)

comparison = pd.concat([minilm_sum, mpnet_sum], ignore_index=True)

# alpha -> readable label
ALPHA_LABELS = {0.0: "BM25 (keyword)", 0.7: "Hibrid", 1.0: "Semantic"}
comparison["method"] = comparison["alpha"].map(ALPHA_LABELS)

pivot = comparison.pivot_table(
    index="method",
    columns="chunk_size",
    values=["P@5", "P@10", "coverage@10", "avg_docs@10"],
    aggfunc="first"
)
pivot = pivot.reindex(["BM25 (keyword)", "Hibrid", "Semantic"])

print("\nFinal comparison (Precision@K over retrieved unique papers after deduplication):")
display(pivot)


Queries with < 10 unique docs (224-token):


,chunk_size,alpha,query,P@5,P@10,coverage@10,n_docs@10
2,224-token,0.00,explainable ai evaluation,0.40,0.56,0.90,9
4,224-token,0.00,fairness explanation,0.40,0.33,0.90,9
5,224-token,0.00,feature attribution,0.80,0.75,0.80,8
7,224-token,0.00,model debugging,0.20,0.12,0.80,8
9,224-token,0.00,post hoc explanation,1.00,0.89,0.90,9
12,224-token,0.70,explainable ai evaluation,0.80,0.78,0.90,9
13,224-token,0.70,explanation methods,1.00,0.88,0.80,8
14,224-token,0.70,fairness explanation,0.60,0.33,0.90,9
15,224-token,0.70,feature attribution,0.80,0.75,0.80,8
17,224-token,0.70,model debugging,0.40,0.50,0.60,6



Queries with < 10 unique docs (360-token):


,chunk_size,alpha,query,P@5,P@10,coverage@10,n_docs@10
7,360-token,0.00,model debugging,0.20,0.22,0.90,9
8,360-token,0.00,model interpretability,1.00,1.00,0.90,9
10,360-token,0.70,causal explanation,0.40,0.44,0.90,9
11,360-token,0.70,counterfactual explanation,0.40,0.22,0.90,9
13,360-token,0.70,explanation methods,1.00,1.00,0.90,9
17,360-token,0.70,model debugging,0.60,0.56,0.90,9
20,360-token,1.00,causal explanation,0.60,0.56,0.90,9
21,360-token,1.00,counterfactual explanation,0.20,0.11,0.90,9
22,360-token,1.00,explainable ai evaluation,0.80,0.67,0.90,9
23,360-token,1.00,explanation methods,1.00,1.00,0.90,9



All queries sorted by n_docs (224-token):


...

,chunk_size,alpha,P@5,P@10,coverage@10,avg_docs@10,min_docs@10
0,224-token,0.00,0.70,0.62,0.93,9.30,8
1,224-token,0.70,0.70,0.62,0.87,8.70,6
2,224-token,1.00,0.54,0.52,0.91,9.10,7



360-token index summary (MPNet run):


,chunk_size,alpha,P@5,P@10,coverage@10,avg_docs@10,min_docs@10
0,360-token,0.00,0.74,0.65,0.98,9.80,9
1,360-token,0.70,0.72,0.68,0.96,9.60,9
2,360-token,1.00,0.68,0.65,0.94,9.40,9



Final comparison (Precision@K over retrieved unique papers after deduplication):


P@10                 P@5           avg_docs@10            \
chunk_size     224-token 360-token 224-token 360-token   224-token 360-token   
method                                                                         
BM25 (keyword)      0.62      0.65      0.70      0.74        9.30      9.80   
Hibrid              0.62      0.68      0.70      0.72        8.70      9.60   
Semantic            0.52      0.65      0.54      0.68        9.10      9.40   

               coverage@10            
chunk_size       224-token 360-token  
method                                
BM25 (keyword)        0.93      0.98  
Hibrid                0.87      0.96  
Semantic              0.91      0.94

**Key findings:**
- Keyword search (BM25): Both chunk sizes perform similarly (224-token: 0.62, 360-token: 0.65). This is expected, as BM25 matches exact terms rather than semantic meaning. The small difference comes from chunk size affecting which text BM25 sees.
- Hybrid search: Larger chunks work better (0.68 vs 0.62 for P@10). Combining semantic and keyword search helps, and the 360-token chunks give embeddings more context to work with.
- Semantic search: Biggest difference here (0.65 vs 0.52). Pure embedding search really benefits from larger chunks, because having more text per chunk allows the model to better understand what a paper is about.

**Why is hybrid better than pure semantic search?**
When you search by embeddings alone, the system finds papers that are topically related, but might not answer your specific question. Hybrid search combines the best of both:
- Keywords ensure the paper actually contains the terms we're looking for
- Embeddings catch papers that phrase things differently but mean the same thing

Together, these give more precise results than either method alone.

**Why does chunk size matter?**
Smaller chunks mean more chunks per paper. If the search returns multiple chunks from the same paper in the top 10 chunk results, we deduplicate them, keeping only the best chunk from each paper. This is why 224-token chunks sometimes give us only 8-9 unique papers instead of 10 (avg 8.7 at hybrid), while 360-token chunks usually give 9-10 (avg 9.6).

**Bottom line:**
Larger chunks (360-token) work better across all search modes. The best setup is hybrid search with 360-token chunks. We get precise keyword matching plus semantic understanding, and fewer papers get "lost" to chunk duplication.